# Task 2.4 — Silver: Dropout Risk Classification
## Notebook: 07_silver_dropout_risk

**Layer:** Silver (classification from engagement summary)

**What this notebook does:**
- Reads silver.student_course_engagement (Task 2.3 output)
- Applies business rules to classify each enrollment into HIGH / MEDIUM / LOW risk
- Computes risk_scored_date to track when the scoring happened
- Performs longitudinal tracking: compares current tier vs previous week's tier
- Flags students whose risk tier worsened week-over-week
- Writes to silver.enrollment_risk_profile

**Source table:** mini_project_grp6.silver.student_course_engagement
**Target table:** mini_project_grp6.silver.enrollment_risk_profile

**Business Rules (from spec):**
- HIGH   : days_since_last_active > 14 AND current_progress_pct < 30
- MEDIUM : (days_since_last_active > 7) OR (quiz_pass_rate < 0.40) OR
           (video_completion_rate < 0.25)
- LOW    : all others (ACTIVE students who don't meet HIGH or MEDIUM)
- Only ACTIVE enrollments are risk-scored
  (COMPLETED / DROPPED / PAUSED get their own tier label, not scored)

**Run order:** Run all cells top-to-bottom. Safe to re-run (OVERWRITE mode).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
SILVER  = "silver"

# --- Source and target table names (fully qualified) ---
ENGAGEMENT_SILVER_TABLE  = f"{CATALOG}.{SILVER}.student_course_engagement"
RISK_PROFILE_TABLE       = f"{CATALOG}.{SILVER}.enrollment_risk_profile"
DQ_AUDIT_TABLE           = f"{CATALOG}.audit.dq_audit"

# --- Dropout risk threshold values (from project spec) ---
HIGH_DAYS_INACTIVE_THRESHOLD  = 14     # days_since_last_active > 14
HIGH_PROGRESS_THRESHOLD       = 30.0   # current_progress_pct < 30

MEDIUM_DAYS_INACTIVE_THRESHOLD = 7     # days_since_last_active > 7
MEDIUM_QUIZ_PASS_THRESHOLD     = 0.40  # quiz_pass_rate < 0.40
MEDIUM_VIDEO_COMPLETION_THRESHOLD = 0.25  # video_completion_rate < 0.25

# --- Risk tier labels ---
TIER_HIGH   = "HIGH"
TIER_MEDIUM = "MEDIUM"
TIER_LOW    = "LOW"

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(SILVER)

print(f"✅ Catalog  : {CATALOG}")
print(f"✅ Schema   : {SILVER}")
print(f"✅ Source   : {ENGAGEMENT_SILVER_TABLE}")
print(f"✅ Target   : {RISK_PROFILE_TABLE}")
print()
print("Risk thresholds:")
print(f"   HIGH   — inactive > {HIGH_DAYS_INACTIVE_THRESHOLD}d AND progress < {HIGH_PROGRESS_THRESHOLD}%")
print(f"   MEDIUM — inactive > {MEDIUM_DAYS_INACTIVE_THRESHOLD}d OR quiz_pass < {MEDIUM_QUIZ_PASS_THRESHOLD} OR video < {MEDIUM_VIDEO_COMPLETION_THRESHOLD}")
print(f"   LOW    — all other ACTIVE enrollments")

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType,
    BooleanType, DateType, TimestampType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Read Source + Validate

Read silver.student_course_engagement and validate before classification.
We only score ACTIVE enrollments for dropout risk.
COMPLETED, DROPPED, PAUSED get a separate tier label so they are
still present in the output table but not mixed with scored students.

In [0]:
# ============================================================
# CELL 5 — Read silver.student_course_engagement and validate
# ============================================================

eng_df = spark.table(ENGAGEMENT_SILVER_TABLE)

total_rows     = eng_df.count()
active_count   = eng_df.filter(F.col("enrollment_status") == "ACTIVE").count()
completed      = eng_df.filter(F.col("enrollment_status") == "COMPLETED").count()
dropped        = eng_df.filter(F.col("enrollment_status") == "DROPPED").count()
paused         = eng_df.filter(F.col("enrollment_status") == "PAUSED").count()

print("── Source validation: student_course_engagement ──")
print(f"   Total rows        : {total_rows:,}")
print(f"   ACTIVE (to score) : {active_count:,}")
print(f"   COMPLETED         : {completed:,}")
print(f"   DROPPED           : {dropped:,}")
print(f"   PAUSED            : {paused:,}")
print()

# Confirm all required columns exist
required_cols = [
    "student_id", "course_id", "enrollment_status",
    "days_since_last_active", "current_progress_pct",
    "quiz_pass_rate", "video_completion_rate",
    "engagement_trend", "last_active_date"
]
missing = [c for c in required_cols if c not in eng_df.columns]
if missing:
    raise ValueError(f"❌ Missing required columns: {missing}")

print(f"✅ All {len(required_cols)} required columns present")
print()

# Preview distribution of key risk signal fields for ACTIVE students
print("── Key signals for ACTIVE students ───────────────")
eng_df.filter(F.col("enrollment_status") == "ACTIVE").select(
    F.round(F.avg("days_since_last_active"), 2).alias("avg_days_inactive"),
    F.round(F.avg("current_progress_pct"), 2).alias("avg_progress_pct"),
    F.round(F.avg("quiz_pass_rate"), 4).alias("avg_quiz_pass_rate"),
    F.round(F.avg("video_completion_rate"), 4).alias("avg_video_completion")
).show()

## Part B — Apply Dropout Risk Classification Rules

Classification logic (applied in order — HIGH takes precedence over MEDIUM):

  HIGH   : days_since_last_active > 14  AND  current_progress_pct < 30
  MEDIUM : (days_since_last_active > 7) OR (quiz_pass_rate < 0.40)
                                         OR (video_completion_rate < 0.25)
  LOW    : all active students not meeting HIGH or MEDIUM criteria

Non-ACTIVE enrollments:
  COMPLETED → tier = "COMPLETED" (not a risk, out of scope)
  DROPPED   → tier = "DROPPED"   (already churned)
  PAUSED    → tier = "PAUSED"    (on hold, monitor separately)

We also add three boolean "reason" columns so instructors can see
exactly which rule(s) triggered the risk tier for each student.
This makes the output actionable, not just a label.

In [0]:
# ============================================================
# CELL 7 — Apply HIGH / MEDIUM / LOW classification
#
# Step 1: add boolean columns for each individual risk condition
#         (these become the "reason flags" visible in dashboards)
# Step 2: apply tier hierarchy using nested F.when()
#         HIGH check first → MEDIUM check → LOW fallback
# Step 3: non-ACTIVE enrollments get their status as the tier label
#
# The three condition columns:
#   is_high_inactive  = days_since_last_active > 14
#   is_low_progress   = current_progress_pct < 30
#   is_low_quiz_pass  = quiz_pass_rate < 0.40
#   is_low_video      = video_completion_rate < 0.25
#   is_medium_inactive= days_since_last_active > 7
# ============================================================

risk_scored_df = (
    eng_df
    # ── Step 1: Add individual condition flags ───────────────
    .withColumn(
        "is_high_inactive",
        F.col("days_since_last_active") > F.lit(HIGH_DAYS_INACTIVE_THRESHOLD)
    )
    .withColumn(
        "is_low_progress",
        F.col("current_progress_pct") < F.lit(HIGH_PROGRESS_THRESHOLD)
    )
    .withColumn(
        "is_medium_inactive",
        F.col("days_since_last_active") > F.lit(MEDIUM_DAYS_INACTIVE_THRESHOLD)
    )
    .withColumn(
        "is_low_quiz_pass",
        F.col("quiz_pass_rate") < F.lit(MEDIUM_QUIZ_PASS_THRESHOLD)
    )
    .withColumn(
        "is_low_video_completion",
        F.col("video_completion_rate") < F.lit(MEDIUM_VIDEO_COMPLETION_THRESHOLD)
    )
    # ── Step 2: Apply tier hierarchy ─────────────────────────
    .withColumn(
        "dropout_risk_tier",
        F.when(
            # Non-ACTIVE enrollments get their status as tier
            F.col("enrollment_status") == "COMPLETED",
            F.lit("COMPLETED")
        ).when(
            F.col("enrollment_status") == "DROPPED",
            F.lit("DROPPED")
        ).when(
            F.col("enrollment_status") == "PAUSED",
            F.lit("PAUSED")
        ).when(
            # HIGH: inactive > 14d AND progress < 30% (both conditions required)
            (F.col("is_high_inactive")) & (F.col("is_low_progress")),
            F.lit(TIER_HIGH)
        ).when(
            # MEDIUM: any one of the three conditions
            (F.col("is_medium_inactive")) |
            (F.col("is_low_quiz_pass")) |
            (F.col("is_low_video_completion")),
            F.lit(TIER_MEDIUM)
        ).otherwise(
            # LOW: active students who pass all checks
            F.lit(TIER_LOW)
        )
    )
    # ── Step 3: Add scoring timestamp ────────────────────────
    .withColumn("risk_scored_date", F.current_date())
)

print("── Risk tier distribution ────────────────────────")
risk_scored_df.groupBy("dropout_risk_tier").count().orderBy("dropout_risk_tier").show()

print("── Active students only ──────────────────────────")
risk_scored_df.filter(
    F.col("enrollment_status") == "ACTIVE"
).groupBy("dropout_risk_tier").count().orderBy("dropout_risk_tier").show()

## Part C — Longitudinal Tracking (Week-over-Week Tier Comparison)

To detect students who are getting worse over time, we compare
the current risk tier to the previous week's tier stored in the
existing enrollment_risk_profile table (if it exists).

Worsening is defined as:
  LOW  → MEDIUM  (tier_worsened = true)
  LOW  → HIGH    (tier_worsened = true)
  MEDIUM → HIGH  (tier_worsened = true)

On first run (table does not yet exist), all students get
prev_week_tier = NULL and tier_worsened = false.

In [0]:
# ============================================================
# CELL 9 — Load previous week's risk tier for each student-course
#
# On first run: table doesn't exist → prev_week_tier = NULL for all rows
# On subsequent runs: read the most recent dropout_risk_tier
#                     stored from the last run for each student-course pair
#
# We store only student_id, course_id, and the previous tier
# to join back later.
# ============================================================

table_exists = spark.catalog.tableExists(RISK_PROFILE_TABLE)

if table_exists:
    prev_tiers_df = (
        spark.table(RISK_PROFILE_TABLE)
        .filter(F.col("enrollment_status") == "ACTIVE")   # only compare active enrollments
        .select(
            "student_id",
            "course_id",
            F.col("dropout_risk_tier").alias("prev_week_tier"),
            F.col("risk_scored_date").alias("prev_scored_date")
        )
    )
    prev_count = prev_tiers_df.count()
    print(f"✅ Previous tier table found — {prev_count:,} rows loaded for comparison")
    print()
    print("── Previous week tier distribution ───────────────")
    prev_tiers_df.groupBy("prev_week_tier").count().orderBy("prev_week_tier").show()
else:
    # First run — create an empty DataFrame with the same schema
    from pyspark.sql.types import StructType, StructField, StringType, DateType
    empty_schema = StructType([
        StructField("student_id",      StringType(), True),
        StructField("course_id",       StringType(), True),
        StructField("prev_week_tier",  StringType(), True),
        StructField("prev_scored_date", DateType(),  True),
    ])
    prev_tiers_df = spark.createDataFrame([], empty_schema)
    print("ℹ  First run — no previous tier table found")
    print("   prev_week_tier will be NULL for all rows")

In [0]:
# ============================================================
# CELL 10 — Join previous week's tier and compute tier_worsened
#
# Tier ordering for worsening detection (numerically):
#   LOW    = 1  (best)
#   MEDIUM = 2
#   HIGH   = 3  (worst)
#
# tier_worsened = true if current tier rank > previous tier rank
#
# Non-ACTIVE statuses (COMPLETED, DROPPED, PAUSED) are excluded
# from worsening logic — they are not risk-scored.
# ============================================================

# Numeric rank map for tier comparison
def tier_rank(col_name):
    return (
        F.when(F.col(col_name) == "LOW",    F.lit(1))
         .when(F.col(col_name) == "MEDIUM", F.lit(2))
         .when(F.col(col_name) == "HIGH",   F.lit(3))
         .otherwise(F.lit(None).cast(IntegerType()))   # NULL for non-risk tiers
    )

# Join previous tier onto current scored DataFrame
risk_with_history_df = (
    risk_scored_df
    .join(
        prev_tiers_df.select("student_id", "course_id",
                             "prev_week_tier", "prev_scored_date"),
        on=["student_id", "course_id"],
        how="left"
    )
    # Add numeric ranks for comparison
    .withColumn("current_tier_rank",  tier_rank("dropout_risk_tier"))
    .withColumn("previous_tier_rank", tier_rank("prev_week_tier"))
    # tier_worsened = current rank > previous rank (only for ACTIVE enrollments)
    .withColumn(
        "tier_worsened",
        F.when(
            (F.col("enrollment_status") == "ACTIVE") &
            (F.col("previous_tier_rank").isNotNull()) &
            (F.col("current_tier_rank") > F.col("previous_tier_rank")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    # Drop temporary rank columns
    .drop("current_tier_rank", "previous_tier_rank")
)

worsened_count = risk_with_history_df.filter(F.col("tier_worsened") == True).count()
print("── Tier worsening analysis ───────────────────────")
print(f"   Students whose tier worsened this week: {worsened_count:,}")
print()

if worsened_count > 0:
    print("── Worsened students breakdown ───────────────────")
    risk_with_history_df.filter(F.col("tier_worsened") == True) \
        .groupBy("prev_week_tier", "dropout_risk_tier") \
        .count() \
        .orderBy("prev_week_tier", "dropout_risk_tier") \
        .show(truncate=False)
else:
    print("   ℹ  No tier worsening detected (or this is the first run)")

In [0]:
# ============================================================
# CELL 11 — Assemble final Silver DataFrame
#           Select all output columns in clean order
#           Include all engagement features used in classification
#           (spec requires: "include all engagement features used")
#           Add Silver metadata columns
# ============================================================

enrollment_risk_df = (
    risk_with_history_df
    .select(
        # ── Primary keys ──────────────────────────────────────
        "student_id",
        "course_id",
        "enrollment_status",
        # ── Risk classification output ────────────────────────
        "dropout_risk_tier",
        "risk_scored_date",
        # ── Longitudinal tracking ─────────────────────────────
        "prev_week_tier",
        "prev_scored_date",
        "tier_worsened",
        # ── All engagement features used in classification ────
        # (included per spec: "include all engagement features used")
        "days_since_last_active",
        "current_progress_pct",
        "quiz_pass_rate",
        "video_completion_rate",
        # ── Individual risk condition flags (for transparency) ─
        "is_high_inactive",
        "is_low_progress",
        "is_medium_inactive",
        "is_low_quiz_pass",
        "is_low_video_completion",
        # ── Additional engagement context ─────────────────────
        "total_video_minutes_watched",
        "quiz_attempts_count",
        "discussion_posts_count",
        "login_count_7d",
        "login_count_30d",
        "last_active_date",
        "engagement_trend",
    )
    # Add Silver metadata
    .withColumn("_silver_load_ts", F.current_timestamp())
    .withColumn("_schema_version",  F.lit(SCHEMA_VERSION))
)

print("── Final Risk Profile DataFrame ──────────────────")
print(f"   Rows    : {enrollment_risk_df.count():,}")
print(f"   Columns : {len(enrollment_risk_df.columns)}")
print()
enrollment_risk_df.printSchema()

In [0]:
# ============================================================
# CELL 12 — Write to silver.enrollment_risk_profile
#           Mode: OVERWRITE — fully recomputed on every daily run
#           The previous tier comparison is done BEFORE overwrite (Cell 9-10)
#           so we always have longitudinal tracking even with OVERWRITE
# ============================================================

(
    enrollment_risk_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RISK_PROFILE_TABLE)
)

final_count = spark.table(RISK_PROFILE_TABLE).count()
print(f"✅ Written to: {RISK_PROFILE_TABLE}")
print(f"   Rows in table: {final_count:,}")

In [0]:
# ============================================================
# CELL 13 — Verify silver.enrollment_risk_profile
# ============================================================

risk_df = spark.table(RISK_PROFILE_TABLE)

print("── enrollment_risk_profile ───────────────────────")
print(f"Total rows   : {risk_df.count():,}")
print(f"Columns      : {len(risk_df.columns)}")
print()

print("── Full tier distribution ────────────────────────")
risk_df.groupBy("dropout_risk_tier").count().orderBy("dropout_risk_tier").show()

print("── ACTIVE students only — risk tiers ────────────")
risk_df.filter(F.col("enrollment_status") == "ACTIVE") \
    .groupBy("dropout_risk_tier").count() \
    .orderBy("dropout_risk_tier").show()

print("── Tier worsened breakdown ───────────────────────")
risk_df.groupBy("tier_worsened").count().show()

print("── HIGH risk students sample ─────────────────────")
risk_df.filter(F.col("dropout_risk_tier") == "HIGH") \
    .select(
        "student_id", "course_id",
        "days_since_last_active", "current_progress_pct",
        "quiz_pass_rate", "video_completion_rate",
        "engagement_trend", "prev_week_tier", "tier_worsened"
    ).show(10, truncate=False)

## Part D — Data Quality Checks (Silver)

DQ checks for enrollment_risk_profile:
1. dropout_risk_tier must be one of the valid values
2. risk_scored_date must not be NULL
3. student_id and course_id must not be NULL
4. is_high_inactive AND is_low_progress must both be true for all HIGH tier rows
   (verify that the classification logic applied correctly)
5. tier_worsened must be false for all non-ACTIVE enrollments

In [0]:
# ============================================================
# CELL 15 — DQ Check 1: dropout_risk_tier must be one of valid values
# Valid: HIGH, MEDIUM, LOW, COMPLETED, DROPPED, PAUSED
# ============================================================

risk_df = spark.table(RISK_PROFILE_TABLE)

valid_tiers = ["HIGH", "MEDIUM", "LOW", "COMPLETED", "DROPPED", "PAUSED"]

invalid_tier = (
    risk_df
    .filter(~F.col("dropout_risk_tier").isin(valid_tiers))
    .withColumn("dq_check_name", F.lit("invalid_dropout_risk_tier"))
    .withColumn("dq_table",      F.lit(RISK_PROFILE_TABLE))
    .withColumn("dq_severity",   F.lit("HIGH"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("dropout_risk_tier has invalid value:"),
        F.col("dropout_risk_tier"),
        F.lit("student_id:"), F.col("student_id")
    ))
)

invalid_count = invalid_tier.count()
print("── DQ Check 1: dropout_risk_tier valid values ────")
if invalid_count > 0:
    (
        invalid_tier
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "student_id", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_count} rows with invalid tier → {DQ_AUDIT_TABLE}")
    invalid_tier.groupBy("dropout_risk_tier").count().show()
else:
    print(f"   ✅ PASSED — all tiers are one of: {valid_tiers}")

In [0]:
# ============================================================
# CELL 16 — DQ Check 2: NULL student_id and course_id
#           These are the primary keys of the risk profile table
# ============================================================

risk_df = spark.table(RISK_PROFILE_TABLE)
total   = risk_df.count()

for col_name in ["student_id", "course_id"]:
    null_count = risk_df.filter(F.col(col_name).isNull()).count()
    pct        = (null_count / total * 100) if total > 0 else 0
    status     = "✅" if null_count == 0 else "⚠"
    alert      = "  ← ALERT: exceeds 0.5% threshold!" if pct > 0.5 else ""

    print(f"── DQ Check 2: NULL {col_name} ─────────────────")
    print(f"   {status}  nulls={null_count:,}  ({pct:.3f}%){alert}")

    if null_count > 0:
        flagged = (
            risk_df.filter(F.col(col_name).isNull())
            .withColumn("dq_check_name", F.lit(f"null_{col_name}_risk_profile"))
            .withColumn("dq_table",      F.lit(RISK_PROFILE_TABLE))
            .withColumn("dq_severity",   F.lit("HIGH"))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.lit(f"{col_name} is NULL in enrollment_risk_profile"))
        )
        (
            flagged
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "student_id", "course_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {null_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    print()

In [0]:
# ============================================================
# CELL 17 — DQ Check 3: Classification logic verification
#           Every HIGH tier row must have:
#             is_high_inactive = true AND is_low_progress = true
#           If any HIGH row does NOT satisfy both conditions,
#           the F.when() logic in Cell 7 has a bug.
#
#           Every LOW tier ACTIVE row must NOT satisfy
#           any MEDIUM condition — spot check.
# ============================================================

risk_df = spark.table(RISK_PROFILE_TABLE)

# ── HIGH tier verification ─────────────────────────────────
high_rows = risk_df.filter(
    (F.col("dropout_risk_tier") == "HIGH") &
    (F.col("enrollment_status") == "ACTIVE")
)
high_total = high_rows.count()

# All HIGH rows must have both flags true
high_invalid = high_rows.filter(
    (~F.col("is_high_inactive")) | (~F.col("is_low_progress"))
).count()

print("── DQ Check 3a: HIGH tier logic verification ─────")
print(f"   Total HIGH tier ACTIVE rows : {high_total:,}")
if high_invalid > 0:
    print(f"   ⚠ {high_invalid} HIGH rows do NOT satisfy both conditions — classification bug!")
else:
    print(f"   ✅ PASSED — all HIGH rows have is_high_inactive=true AND is_low_progress=true")

print()

# ── MEDIUM tier verification ───────────────────────────────
medium_rows = risk_df.filter(
    (F.col("dropout_risk_tier") == "MEDIUM") &
    (F.col("enrollment_status") == "ACTIVE")
)
medium_total = medium_rows.count()

# All MEDIUM rows must satisfy at least one MEDIUM condition
# AND must NOT satisfy the HIGH condition (both flags)
medium_invalid = medium_rows.filter(
    (~F.col("is_medium_inactive")) &
    (~F.col("is_low_quiz_pass")) &
    (~F.col("is_low_video_completion"))
).count()

print("── DQ Check 3b: MEDIUM tier logic verification ───")
print(f"   Total MEDIUM tier ACTIVE rows: {medium_total:,}")
if medium_invalid > 0:
    print(f"   ⚠ {medium_invalid} MEDIUM rows do NOT satisfy any MEDIUM condition — classification bug!")
else:
    print(f"   ✅ PASSED — all MEDIUM rows satisfy at least one MEDIUM condition")

In [0]:
# ============================================================
# CELL 18 — DQ Check 4: tier_worsened must be false for
#           non-ACTIVE enrollments (COMPLETED, DROPPED, PAUSED)
#           These statuses are not risk-scored so worsening
#           flag should never be true for them
# ============================================================

risk_df = spark.table(RISK_PROFILE_TABLE)

invalid_worsened = (
    risk_df
    .filter(
        (F.col("enrollment_status") != "ACTIVE") &
        (F.col("tier_worsened") == True)
    )
    .withColumn("dq_check_name", F.lit("tier_worsened_on_non_active"))
    .withColumn("dq_table",      F.lit(RISK_PROFILE_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("tier_worsened=true for non-ACTIVE enrollment status:"),
        F.col("enrollment_status"),
        F.lit("student_id:"), F.col("student_id")
    ))
)

invalid_worsened_count = invalid_worsened.count()
print("── DQ Check 4: tier_worsened on non-ACTIVE rows ──")
if invalid_worsened_count > 0:
    (
        invalid_worsened
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "student_id", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_worsened_count} rows flagged → {DQ_AUDIT_TABLE}")
    invalid_worsened.select(
        "student_id", "course_id", "enrollment_status", "tier_worsened"
    ).show(5, truncate=False)
else:
    print("   ✅ PASSED — tier_worsened is false for all non-ACTIVE enrollments")

In [0]:
%sql
-- ============================================================
-- CELL 19 — SQL verification of silver.enrollment_risk_profile
-- ============================================================

SELECT
    dropout_risk_tier,
    enrollment_status,
    COUNT(*)                                           AS row_count,
    ROUND(AVG(days_since_last_active), 2)              AS avg_days_inactive,
    ROUND(AVG(current_progress_pct), 2)                AS avg_progress_pct,
    ROUND(AVG(quiz_pass_rate), 4)                      AS avg_quiz_pass_rate,
    ROUND(AVG(video_completion_rate), 4)               AS avg_video_completion,
    SUM(CASE WHEN tier_worsened = true THEN 1 ELSE 0 END) AS worsened_count
FROM mini_project_grp6.silver.enrollment_risk_profile
GROUP BY dropout_risk_tier, enrollment_status
ORDER BY dropout_risk_tier, enrollment_status;

In [0]:
%sql
-- ============================================================
-- CELL 20 — Students whose risk tier worsened this week
--           These are priority students for instructor outreach
-- ============================================================

SELECT
    student_id,
    course_id,
    prev_week_tier,
    dropout_risk_tier            AS current_tier,
    days_since_last_active,
    current_progress_pct,
    quiz_pass_rate,
    video_completion_rate,
    engagement_trend,
    risk_scored_date
FROM mini_project_grp6.silver.enrollment_risk_profile
WHERE tier_worsened = true
  AND enrollment_status = 'ACTIVE'
ORDER BY
    CASE dropout_risk_tier WHEN 'HIGH' THEN 1 WHEN 'MEDIUM' THEN 2 ELSE 3 END,
    days_since_last_active DESC;

In [0]:
%sql
-- ============================================================
-- CELL 21 — Lead time check: are we flagging HIGH risk students
--           at least 14 days before their expected completion date?
--           This validates the core business success metric.
-- ============================================================

SELECT
    r.student_id,
    r.course_id,
    r.dropout_risk_tier,
    r.days_since_last_active,
    r.current_progress_pct,
    r.risk_scored_date,
    e.expected_completion_date,
    DATEDIFF(e.expected_completion_date, r.risk_scored_date) AS days_until_expected_completion
FROM mini_project_grp6.silver.enrollment_risk_profile r
JOIN mini_project_grp6.bronze.enrollments_bronze e
    ON r.student_id = e.student_id
   AND r.course_id  = e.course_id
WHERE r.dropout_risk_tier = 'HIGH'
  AND r.enrollment_status = 'ACTIVE'
ORDER BY days_until_expected_completion DESC
LIMIT 20;

In [0]:
# ============================================================
# CELL 22 — Task 2.4 completion summary
# ============================================================

final_df      = spark.table(RISK_PROFILE_TABLE)
final_count   = final_df.count()
active_df     = final_df.filter(F.col("enrollment_status") == "ACTIVE")

high_count    = active_df.filter(F.col("dropout_risk_tier") == "HIGH").count()
medium_count  = active_df.filter(F.col("dropout_risk_tier") == "MEDIUM").count()
low_count     = active_df.filter(F.col("dropout_risk_tier") == "LOW").count()
worsened      = final_df.filter(F.col("tier_worsened") == True).count()

print("═" * 60)
print("  TASK 2.4 COMPLETE — Silver Dropout Risk Profile")
print("═" * 60)
print()
print(f"  ✅ {RISK_PROFILE_TABLE}")
print(f"      Total rows              : {final_count:,}")
print(f"      Mode                    : OVERWRITE (fully recomputed)")
print()
print("  ACTIVE enrollment risk tiers:")
print(f"      🔴 HIGH   : {high_count:,}  (inactive>{HIGH_DAYS_INACTIVE_THRESHOLD}d AND progress<{HIGH_PROGRESS_THRESHOLD}%)")
print(f"      🟡 MEDIUM : {medium_count:,}  (inactive>{MEDIUM_DAYS_INACTIVE_THRESHOLD}d OR quiz<{MEDIUM_QUIZ_PASS_THRESHOLD} OR video<{MEDIUM_VIDEO_COMPLETION_THRESHOLD})")
print(f"      🟢 LOW    : {low_count:,}  (all other active)")
print()
print(f"  Longitudinal tracking:")
print(f"      Students with worsened tier this week: {worsened:,}")
print(f"      prev_week_tier stored for next run comparison")
print()
print("  Condition flags in output (for transparency):")
print("      is_high_inactive, is_low_progress")
print("      is_medium_inactive, is_low_quiz_pass, is_low_video_completion")
print()
print("  DQ checks run:")
print("      Cell 15 — dropout_risk_tier valid values")
print("      Cell 16 — NULL student_id / course_id")
print("      Cell 17 — HIGH/MEDIUM logic verification")
print("      Cell 18 — tier_worsened false on non-ACTIVE rows")
print()
print("  ─────────────────────────────────────────────────────")
print("  ✅ FULL SILVER LAYER COMPLETE — all 4 tables done")
print()
print("  Silver tables:")
print("      mini_project_grp6.silver.discussion_posts_parsed")
print("      mini_project_grp6.silver.learning_sessions")
print("      mini_project_grp6.silver.student_course_engagement")
print("      mini_project_grp6.silver.enrollment_risk_profile")
print()
print("  Next: Task 3.1 → 08_gold_course_performance_daily")
print("         course_id + report_date · PARTITION + Z-ORDER")
print("═" * 60)